In [ ]:
# 필요한 라이브러리 로드
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
import os
print(os.getcwd())

In [ ]:
# 데이터 불러오기 (URL 기반 분류 문제)
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')

In [ ]:
# 데이터 확인
train.head(5)
test.head(5)

In [ ]:
#전처리
train['URL'] = train['URL'].str.replace('[.]','.', regex=False)
test['URL'] = test['URL'].str.replace('[.]','.',regex=False)        # test도 전처리 미리 함

In [ ]:
train.head(), test.head()

In [ ]:
# TF-IDF 벡터화 (char 기반 3~5 ngram)
# → URL 패턴 (login, verify 등) 잡기 위해 사용
tfidf = TfidfVectorizer(analyzer='char',ngram_range=(3,5))
X = tfidf.fit_transform(train['URL'])
# 정답(label) 분리
y = train['label']
# test 데이터도 동일한 tfidf 기준으로 변환
tesx_X = tfidf.transform(test['URL'])

In [ ]:
# train/validation 분리 (성능 확인용)
train_X, val_X, train_y, val_y = train_test_split(
    X, y, test_size=0.2, random_state=2026
)

In [ ]:
from scipy.sparse import save_npz

save_npz('X_train.npz', train_X)
save_npz('X_val.npz', val_X)
train_y.to_csv('y_train.csv', index=False)
val_y.to_csv('y_val.csv', index=False)

In [ ]:
# Naive Bayes 모델
# alpha 튜닝 결과 → 0.0001에서 최고 성능 (약 0.956)
model = MultinomialNB(alpha=0.000001)
model.fit(train_X,train_y)
# validation 데이터로 AUC 평가

prob = model.predict_proba(val_X)
roc_auc_score(val_y,prob[:, 1])
# → 확률 기반 성능 평가 (이진 분류)

In [ ]:
print('AUC: ', roc_auc_score(val_y,prob[:,1]))

In [ ]:
# 테스트 데이터 예측 (확률값 사용)
real_pred = model.predict_proba(tesx_X)[:,-1]

In [ ]:
# 제출 파일 생성
submission = pd.read_csv('./data/sample_submission.csv')
submission['probability'] = real_pred
submission.to_csv('answer3.csv',index=False)

In [ ]:
import pickle

with open('tfidf_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

In [ ]:
# =====================
# 실험 기록
# (2,4) → 약 0.93
# (3,5) → 약 0.94
# alpha=0.0001 → 약 0.956 (최고 성능)
# =====================